In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
AZURE_OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_OPENAI_GPT4_DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_GPT4_DEPLOYMENT_NAME")
AZURE_OPENAI_API_VERSION = os.getenv("AZURE_OPENAI_API_VERSION")
APPLICATIONINSIGHTS_CONNECTION_STRING = os.getenv("APPLICATIONINSIGHTS_CONNECTION_STRING")

In [2]:
import os

from azure.identity import DefaultAzureCredential
from promptflow.core import AzureOpenAIModelConfiguration
from promptflow.evals.evaluators import (
    CoherenceEvaluator,
    SelfHarmEvaluator,
)


# Initialize Azure OpenAI Connection with your environment variables
model_config = AzureOpenAIModelConfiguration(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    azure_deployment=AZURE_OPENAI_GPT4_DEPLOYMENT_NAME,
    api_version=AZURE_OPENAI_API_VERSION,
)


In [3]:
from promptflow.tracing import start_trace
# instrument OpenAI
start_trace()

Prompt flow service has started...


In [4]:

coherence_eval = CoherenceEvaluator(model_config)
score = coherence_eval(question="What is the capital of Japan?", answer="The capital of Japan is Tokyo.")
print(score)

You can view the trace detail from the following URL:
http://127.0.0.1:23333/v1.0/ui/traces/?#collection=8-Monitoring and Tracing&uiTraceId=0xf5c1b50589901bd6fec3c6ec915cbfb6
You can view the trace detail from the following URL:
http://127.0.0.1:23333/v1.0/ui/traces/?#collection=8-Monitoring and Tracing&uiTraceId=0x6cacb2243b9cb522be0ee4f245a20306
You can view the trace detail from the following URL:
http://127.0.0.1:23333/v1.0/ui/traces/?#collection=8-Monitoring and Tracing&uiTraceId=0xe47b185287e3523d008334f55949d29f


{'gpt_coherence': 5.0}


In [5]:
from promptflow.tracing import trace

# trace your function
@trace
def test_coherence_evaluator():
    coherence_eval = CoherenceEvaluator(model_config)
    score = coherence_eval(question="What is the capital of Japan?", answer="The capital of Japan is Tokyo.")
    print(score)

test_coherence_evaluator()

{'gpt_coherence': 5.0}


### Configure the trace to be sent to Azure Application Insights

In [6]:
from opentelemetry.instrumentation.requests import RequestsInstrumentor
from azure.monitor.opentelemetry.exporter import AzureMonitorLogExporter, AzureMonitorTraceExporter, AzureMonitorMetricExporter
from opentelemetry.instrumentation.langchain import LangchainInstrumentor
from opentelemetry.instrumentation.logging import LoggingInstrumentor
from opentelemetry.sdk.trace.export import BatchSpanProcessor, ConsoleSpanExporter
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry import trace
from azure.monitor.opentelemetry import configure_azure_monitor
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from opentelemetry.sdk.environment_variables import OTEL_EXPORTER_OTLP_ENDPOINT

collection_name = "llmops-workshop"
# if not isinstance(trace.get_tracer_provider(), TracerProvider):
if not APPLICATIONINSIGHTS_CONNECTION_STRING:
    raise ValueError("APPLICATIONINSIGHTS_CONNECTION_STRING environment variable is not set.")
os.environ["APPLICATIONINSIGHTS_CONNECTION_STRING"] = APPLICATIONINSIGHTS_CONNECTION_STRING
# OTEL_EXPORTER_OTLP_ENDPOINT = APPLICATIONINSIGHTS_CONNECTION_STRING
# setup_exporter_from_environ()

langchain_instrumentor = LangchainInstrumentor()
if not langchain_instrumentor.is_instrumented_by_opentelemetry:
    langchain_instrumentor.instrument()
tracer_provider = TracerProvider()
trace.set_tracer_provider(tracer_provider)
configure_azure_monitor(collection_name=collection_name)
traces_exporter = AzureMonitorTraceExporter()
trace_processor = BatchSpanProcessor(traces_exporter)
tracer_provider.add_span_processor(trace_processor)
RequestsInstrumentor().instrument()

Traceback (most recent call last):
  File "c:\Users\dschlesinger\code\ongoing\advanced-genai-apps\.venv\lib\site-packages\azure\monitor\opentelemetry\_configure.py", line 200, in _setup_instrumentations
    conflict = get_dist_dependency_conflicts(entry_point.dist) # type: ignore
  File "c:\Users\dschlesinger\code\ongoing\advanced-genai-apps\.venv\lib\site-packages\opentelemetry\instrumentation\dependencies.py", line 50, in get_dist_dependency_conflicts
    for dep in dist.requires:
TypeError: 'method' object is not iterable
Traceback (most recent call last):
  File "c:\Users\dschlesinger\code\ongoing\advanced-genai-apps\.venv\lib\site-packages\azure\monitor\opentelemetry\_configure.py", line 200, in _setup_instrumentations
    conflict = get_dist_dependency_conflicts(entry_point.dist) # type: ignore
  File "c:\Users\dschlesinger\code\ongoing\advanced-genai-apps\.venv\lib\site-packages\opentelemetry\instrumentation\dependencies.py", line 50, in get_dist_dependency_conflicts
    for dep

In [7]:
#now the trace should be collected by Azure Application Insights
test_coherence_evaluator()

{'gpt_coherence': 5.0}
